In [1]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [2]:
import warnings
warnings.filterwarnings('ignore')

### Loading package

In [3]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [4]:
from py.utils import verifyDir,verifyFile

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

DATA_PATH = os.getenv('DATA_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
DATA_PATH, MODEL_PATH

('/media/felipe/DATA19/datasets/', '/media/felipe/DATA19/models/')

In [6]:
MODEL_NAME="OneFormer_Swin_Large"
SEG_DATASET="ade20k" # cityscapes

In [7]:
ADE20K_DIR = f"{DATA_PATH}{SEG_DATASET}/"
UPD4K_DIR = f"{DATA_PATH}/upd4k/"

QSCORE_PATH=f"{DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{DATA_PATH}pp2/images/"

SEGMENT_DIR = f"{DATA_PATH}pp2/segmentations/{MODEL_NAME}/{SEG_DATASET}/"
GROUP_PATH = f"{DATA_PATH}pp2/segmentations/{MODEL_NAME}/{SEG_DATASET}_group/"

UPD4k_PATH = f"{DATA_PATH}UrbanPhysicalDisorder/upd4k/"
GROUP_UPD4k_PATH = f"{DATA_PATH}UrbanPhysicalDisorder/upd4k_group/"

In [8]:
verifyDir(GROUP_PATH)
verifyDir(GROUP_UPD4k_PATH)

### Loading data

In [9]:
%%time
from py.datasets import PlacePulse

data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
cities = list(np.sort(data_df["city"].unique()))

CPU times: user 2.41 s, sys: 60.1 ms, total: 2.47 s
Wall time: 581 ms


### Grouping Segmentations and  Disorders

In [10]:
from py.datasets import UrbanPhysicalDisorder

uss = UrbanPhysicalDisorder(data_path=DATA_PATH)
uss.generate_dataset(dataset=f"{SEG_DATASET}_upd4k")

In [11]:
group_df = uss.get_urban_street_categories(by_groups=True)
group_df

,group_name,main_class,class_name,RGB_color,hex_color,isthing,classes,num_classes,group_class_id
0,air_vehicle,airplane,airplane;aeroplane;plane,"(0, 255, 82)",#00FF52,1,[91],1,1
1,animal,animal,animal;animate;being;beast;brute;creature;fauna,"(255, 0, 122)",#FF007A,1,[127],1,2
2,body_water,water,water,"(61, 230, 250)",#3DE6FA,0,"[22, 27, 61, 114, 129]",5,3
3,city_elements,signboard,signboard;sign,"(0, 71, 255)",#0047FF,1,"[44, 88, 94, 137]",4,4
4,clothes_object,clothes,apparel;wearing;apparel;dress;clothes,"(0, 112, 255)",#0070FF,1,"[93, 116]",2,5
5,construction,wall,wall,"(180, 120, 120)",#B47878,0,"[1, 2, 6, 15, 26, 33, 43, 49, 62, 78, 80, 85, ...",14,6
6,floor,floor,floor;flooring,"(140, 140, 140)",#8C8C8C,0,"[4, 7, 12, 14, 53, 95]",6,7
7,human,person,person;individual;someone;somebody;mortal;soul,"(150, 5, 61)",#96053D,1,[13],1,8
8,indoor_object,bed,bed,"(204, 5, 255)",#CC05FF,1,"[8, 11, 16, 19, 20, 23, 24, 25, 28, 29, 31, 32...",73,9
9,miscellaneous,blind,blind;screen,"(0, 61, 255)",#003DFF,0,"[64, 102, 113, 121, 124, 126, 133, 145, 148, 1...",11,10


In [12]:
group_df[-len(uss.upd4k_labels):][["group_name", "RGB_color", "hex_color", "isthing"]].to_csv(f"{UPD4K_DIR}/upd4k_categories.csv", sep=";", index=False)

In [13]:
group_df[["group_name", "RGB_color", "hex_color", "classes", "group_class_id"]].to_csv(f"{UPD4K_DIR}/{SEG_DATASET}_upd4k_groups_info.csv", sep=";", index=False)

### Mapping classes

In [14]:
color_dict = group_df.set_index('group_class_id')['RGB_color'].to_dict()
color_dict

{1: (0, 255, 82),
 2: (255, 0, 122),
 3: (61, 230, 250),
 4: (0, 71, 255),
 5: (0, 112, 255),
 6: (180, 120, 120),
 7: (140, 140, 140),
 8: (150, 5, 61),
 9: (204, 5, 255),
 10: (0, 61, 255),
 11: (143, 255, 140),
 12: (230, 230, 230),
 13: (173, 255, 0),
 14: (71, 0, 255),
 15: (6, 230, 230),
 16: (255, 0, 20),
 17: (255, 214, 0),
 18: (4, 200, 3),
 151: (110, 81, 20),
 152: (61, 245, 61),
 153: (255, 96, 55),
 154: (50, 183, 250),
 155: (250, 250, 55),
 156: (89, 134, 179),
 157: (140, 120, 240),
 158: (115, 51, 128),
 159: (170, 240, 209),
 160: (255, 204, 51),
 161: (204, 51, 102),
 162: (250, 125, 187),
 163: (36, 179, 83)}

In [15]:
# Step 1: Create a mapping dictionary from class value to class_id
mapping = {
    cls: row['group_class_id']
    for _, row in group_df.iterrows()
    for cls in row['classes']
}

# Step 2: Vectorized replacement using numpy
vectorized_map = np.vectorize(lambda x: mapping.get(x, x))  # defaults to x if not found

### Group Segmentations + UPD

In [16]:
%%time

upd_merge_segment_df = pd.DataFrame()

for idx, current_city in enumerate(cities):
    print(f"{idx+1}: Grouping {current_city}...")
    upd4k_masks = np.sort(glob.glob(f'{UPD4k_PATH}/{current_city}/masks/*.pkl'))
    if len(upd4k_masks)==0:
        continue
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/masks/")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/ratios/")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/segmented_images/")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/segmented_images_overlay/")

    if verifyFile(f"{GROUP_UPD4k_PATH}/{current_city}/segmentations.csv"):
        seg_upd4k_df = pd.read_csv(f"{GROUP_UPD4k_PATH}/{current_city}/segmentations.csv", sep=";", low_memory=False)
        upd_merge_segment_df = pd.concat([upd_merge_segment_df, seg_upd4k_df], ignore_index=True)
        upd_merge_segment_df.fillna(0, inplace=True)
        continue
    
    seg_upd4k_df = pd.read_csv(f"{UPD4k_PATH}/{current_city}/segmentations.csv", sep=";", low_memory=False)
    seg_upd4k_df = uss.process(seg_upd4k_df, aggregate_classes=True, filter_features=True, keep_disorder=True)
    seg_upd4k_df.drop(columns=["indoor_object", "outdoor_object", "nature_object"], inplace=True)
    seg_upd4k_df.to_csv(f"{GROUP_UPD4k_PATH}/{current_city}/segmentations.csv", sep=";", index=False)

    upd_merge_segment_df = pd.concat([upd_merge_segment_df, seg_upd4k_df], ignore_index=True)
    upd_merge_segment_df.fillna(0, inplace=True)
    
    for cur_mask in tqdm(upd4k_masks):
        current_id = cur_mask.split("/")[-1]
        image_name = current_id.replace(".pkl", "")
        current_image = Image.open(f'{IMAGES_PATH}/{current_city}/{image_name}.JPG'  ).convert("RGB")

        # UPD mask
        usd_mask = joblib.load(cur_mask)
        usd_new_mask = vectorized_map(usd_mask)
        joblib.dump(usd_new_mask, f"{GROUP_UPD4k_PATH}/{current_city}/masks/{current_id}")

        # Group ratio
        unique, counts = np.unique(usd_new_mask, return_counts=True)
        total = usd_new_mask.size
        proportions = {int(k): float(v / total) * 100 for k, v in zip(unique, counts)}
        df = pd.DataFrame(list(proportions.items()), columns=["group_class_id", "ratio"])
        usd_ratio_df = pd.merge(df, group_df[["group_name", "RGB_color", "hex_color", "group_class_id"]], on="group_class_id", how="left")
        usd_ratio_df.to_csv(f"{GROUP_UPD4k_PATH}/{current_city}/ratios/{image_name}.csv", sep=";", index=False)

        # mask
        usd_new_image = uss.convert_matrix_to_mask(usd_new_mask, color_dict)
        usd_new_image.save(f"{GROUP_UPD4k_PATH}{current_city}/segmented_images/{image_name}.png")

        # overlay
        orig_usd_overlay = Image.blend(current_image, usd_new_image, alpha=0.6)
        orig_usd_overlay.save(f"{GROUP_UPD4k_PATH}/{current_city}/segmented_images_overlay/{image_name}.png")


1: Grouping Amsterdam...
2: Grouping Atlanta...
3: Grouping Bangkok...
4: Grouping Barcelona...
5: Grouping Belo Horizonte...
6: Grouping Berlin...
7: Grouping Boston...
8: Grouping Bratislava...
9: Grouping Bucharest...
10: Grouping Cape Town...
11: Grouping Chicago...
12: Grouping Copenhagen...
13: Grouping Denver...
14: Grouping Dublin...
15: Grouping Gaborone...
16: Grouping Glasgow...
17: Grouping Guadalajara...
18: Grouping Helsinki...
19: Grouping Hong Kong...
20: Grouping Houston...
21: Grouping Johannesburg...
22: Grouping Kiev...
23: Grouping Kyoto...
24: Grouping Lisbon...
25: Grouping London...
26: Grouping Los Angeles...
27: Grouping Madrid...
28: Grouping Melbourne...
29: Grouping Mexico City...
30: Grouping Milan...
31: Grouping Minneapolis...
32: Grouping Montreal...
33: Grouping Moscow...
34: Grouping Munich...
35: Grouping New York...
36: Grouping Paris...
37: Grouping Philadelphia...
38: Grouping Portland...
39: Grouping Prague...
40: Grouping Rio De Janeiro...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3644/3644 [05:45<00:00, 10.55it/s]

41: Grouping Rome...
42: Grouping San Francisco...
43: Grouping Santiago...
44: Grouping Sao Paulo...
45: Grouping Seattle...
46: Grouping Singapore...
47: Grouping Stockholm...
48: Grouping Sydney...
49: Grouping Taipei...
50: Grouping Tel Aviv...
51: Grouping Tokyo...
52: Grouping Toronto...
53: Grouping Valparaiso...
54: Grouping Warsaw...
55: Grouping Washington DC...
56: Grouping Zagreb...
CPU times: user 2min 51s, sys: 4.49 s, total: 2min 55s
Wall time: 5min 45s


In [17]:
upd_merge_segment_df

,image_id,seg_image_path,seg_overlay_image_path,mask_path,ratio_path,city_elements,construction,floor,human,sky,...,car_police,car_taxi,damaged_traffic_sign,garbage_bag,garbage_box,graffiti,homeless,kiosk,overhead_cable,trashcan
0,50f5ea5bfdc9f065f0007ab0,Rio De Janeiro/segmented_images/50f5ea5bfdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ea...,Rio De Janeiro/masks/50f5ea5bfdc9f065f0007ab0.pkl,Rio De Janeiro/ratios/50f5ea5bfdc9f065f0007ab0...,0.241667,10.665833,32.825000,0.005833,27.408333,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,5.370000,0.000000
1,50f5ea5bfdc9f065f0007ab1,Rio De Janeiro/segmented_images/50f5ea5bfdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ea...,Rio De Janeiro/masks/50f5ea5bfdc9f065f0007ab1.pkl,Rio De Janeiro/ratios/50f5ea5bfdc9f065f0007ab1...,0.000000,7.706667,11.911667,0.700833,7.156667,...,0.0,0.0,0.0,0.316667,0.0,0.000000,0.0,0.0,0.000000,0.000000
2,50f5ea5bfdc9f065f0007ab2,Rio De Janeiro/segmented_images/50f5ea5bfdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ea...,Rio De Janeiro/masks/50f5ea5bfdc9f065f0007ab2.pkl,Rio De Janeiro/ratios/50f5ea5bfdc9f065f0007ab2...,1.187500,26.355000,26.700833,0.000000,33.570000,...,0.0,0.0,0.0,0.000000,0.0,1.174167,0.0,0.0,4.291667,1.579167
3,50f5ea5bfdc9f065f0007ab3,Rio De Janeiro/segmented_images/50f5ea5bfdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ea...,Rio De Janeiro/masks/50f5ea5bfdc9f065f0007ab3.pkl,Rio De Janeiro/ratios/50f5ea5bfdc9f065f0007ab3...,0.000000,1.163333,0.421667,0.000000,10.571667,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000
4,50f5ea5bfdc9f065f0007ab4,Rio De Janeiro/segmented_images/50f5ea5bfdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ea...,Rio De Janeiro/masks/50f5ea5bfdc9f065f0007ab4.pkl,Rio De Janeiro/ratios/50f5ea5bfdc9f065f0007ab4...,0.000000,45.927500,1.735000,0.000000,36.885000,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3639,50f5ec44fdc9f065f000890f,Rio De Janeiro/segmented_images/50f5ec44fdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ec...,Rio De Janeiro/masks/50f5ec44fdc9f065f000890f.pkl,Rio De Janeiro/ratios/50f5ec44fdc9f065f000890f...,1.496667,26.320000,23.499167,0.000000,20.913333,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,4.249167,0.000000
3640,50f5ec44fdc9f065f0008910,Rio De Janeiro/segmented_images/50f5ec44fdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ec...,Rio De Janeiro/masks/50f5ec44fdc9f065f0008910.pkl,Rio De Janeiro/ratios/50f5ec44fdc9f065f0008910...,1.461667,36.265000,11.671667,0.000000,15.058333,...,0.0,0.0,0.0,0.077500,0.0,0.000000,0.0,0.0,6.012500,0.000000
3641,50f5ec44fdc9f065f0008911,Rio De Janeiro/segmented_images/50f5ec44fdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ec...,Rio De Janeiro/masks/50f5ec44fdc9f065f0008911.pkl,Rio De Janeiro/ratios/50f5ec44fdc9f065f0008911...,2.005833,36.983333,16.257500,0.000000,15.582500,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,11.570833,0.000000
3642,50f5ec44fdc9f065f0008912,Rio De Janeiro/segmented_images/50f5ec44fdc9f0...,Rio De Janeiro/segmented_images_overlay/50f5ec...,Rio De Janeiro/masks/50f5ec44fdc9f065f0008912.pkl,Rio De Janeiro/ratios/50f5ec44fdc9f065f0008912...,0.380000,13.450833,25.781667,0.130000,9.644167,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,8.115833,0.204167


In [18]:
upd_merge_segment_df.to_csv(f"{GROUP_UPD4k_PATH}/segmentations.csv", sep=";", index=False)